# Figure S2F — Time-Varying Cox Forest Plot: Overall Survival (OS)

Self-contained notebook (no external imports). Time-varying Cox model testing whether irAE onset
(Grade 1+ and Grade 3+, per toxicity) is associated with overall survival, using a time-varying
"exposed" covariate (0 before irAE onset, 1 after) so onset timing doesn't bias the hazard estimate.
`T=0` = first line of therapy start; censoring = last follow-up.

**Scope:** OS only (this notebook does not compute PFS — that's a separate, existing panel).
Results are computed for all three cohorts (LOT-1 ICI, LOT-1 non-ICI, All patients) for QC/reference,
but **only the ICI cohort's forest plot is saved as the panel deliverable** — non-ICI and All are
rendered inline for sanity-checking but not written to disk.

**Outputs:**
- `Forest_Cox_OS_S2F.pdf` — ICI cohort OS forest plot (the panel)
- `Forest_Cox_OS_S2F_Results.csv` — OS HR/CI/p-value table for all three cohorts (QC reference)

**Formatting (Nature compliance):** Arial only (hard-fails if not resolved), `pdf.fonttype=42`,
7pt axis label / 6pt tick labels / 5pt annotation text, no `bbox_inches='tight'` on save.


In [ ]:
import os
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = ['Arial']
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
matplotlib.rcParams['axes.unicode_minus'] = False

import matplotlib.pyplot as plt
from lifelines import CoxTimeVaryingFitter

%matplotlib inline

warnings.filterwarnings('ignore')

# ---- Hard-fail if Arial isn't actually resolved (no silent fallback) ----
import matplotlib.font_manager as fm
_arial_path = fm.findfont('Arial', fallback_to_default=False)
if 'Arial' not in _arial_path:
    raise RuntimeError(
        f"Arial not found -- matplotlib resolved to '{_arial_path}' instead. "
        "Install Arial or update font.sans-serif before rendering this figure."
    )
print(f"Arial resolved to: {_arial_path}")


## Paths

Notebook lives in `figure 2/scripts/`. Data lives in the sibling `figure 2/data/` folder; the panel
PDF and results CSV go to `figure 2/results/supp/S2F_Forest_Cox_OS`.


In [ ]:
NOTEBOOK_DIR = os.getcwd()
FIGURES_DIR = os.path.normpath(os.path.join(NOTEBOOK_DIR, '..', '..', '..'))
DATA_DIR = os.path.join(FIGURES_DIR, 'figures_data', 'figure 2', 'data')
RESULTS_DIR = os.path.normpath(os.path.join(NOTEBOOK_DIR, '..', 'results', 'supp', 'S2F_Forest_Cox_OS'))
os.makedirs(RESULTS_DIR, exist_ok=True)

PROGRESSION_PATH = os.path.join(DATA_DIR, 'table_timeline_radiology_cancer_progression_predictions.csv')
LLM_PATIENT_PATH = os.path.join(DATA_DIR, 'llm_calls_patient_level_84k.csv')
GRADE_PATH = os.path.join(DATA_DIR, 'grade_results_84k_FIXED_FP.csv')
LOT_DATA_PATH = os.path.join(DATA_DIR, 'regimen_lot(in).csv')
COVARS_PATH = os.path.join(DATA_DIR, 'OneDrive_1_8-7-2026', 'llm84k_pneumonitis_grade0_20260630.csv')
PATIENTS_PATH = os.path.join(DATA_DIR, 'Patients.csv')
LLM_BATCH_PATH = os.path.join(DATA_DIR, 'llm_calls_batch_level_84k.csv')

PDF_OUT = os.path.join(RESULTS_DIR, 'Forest_Cox_OS_S2F.pdf')
CSV_OUT = os.path.join(RESULTS_DIR, 'Forest_Cox_OS_S2F.csv')

for p in [PROGRESSION_PATH, LLM_PATIENT_PATH, GRADE_PATH, LOT_DATA_PATH, COVARS_PATH, PATIENTS_PATH, LLM_BATCH_PATH]:
    print(('FOUND   ' if os.path.exists(p) else 'MISSING '), p)


## Constants


In [ ]:
TOXICITY_COLUMNS = ['adrenal_insufficiency', 'colitis', 'hyperthyroidism',
                    'hypothyroidism', 'pneumonitis', 'liver_toxicity']
TOXICITY_DISPLAY = {
    'adrenal_insufficiency': 'Adrenal Insufficiency', 'colitis': 'Colitis',
    'hyperthyroidism': 'Hyperthyroidism', 'hypothyroidism': 'Hypothyroidism',
    'pneumonitis': 'Pneumonitis', 'liver_toxicity': 'Liver Toxicity',
}
GRADE_TIER_LABEL = {'g1': 'Grade 1+', 'g3': 'Grade 3+'}
MIN_EVENTS_KM = 10
_TOX_ORDER = TOXICITY_COLUMNS + ['any_ae']
PROGRESSION_STRATEGY = 'yes_or_intermediate'


def standardize_mrn(mrn_series):
    def _clean(mrn):
        try:
            s = str(mrn).strip().strip("'\"").replace('P-', '').replace('p-', '').replace('MSK-', '')
            digits = re.findall(r'\d+', s)
            return str(int(digits[0])).zfill(8) if digits else None
        except (ValueError, TypeError):
            return None
    return mrn_series.apply(_clean)


def normalize_progression_label(val):
    if pd.isna(val):
        return 'Unknown'
    v = str(val).encode('ascii', errors='ignore').decode('ascii')
    v = ' '.join(v.strip().strip("'\"").lower().split()).replace('_', ' ').replace('-', ' ')
    if 'intermediate' in v or 'indeterminate' in v:
        return 'Intermediate'
    if v in ('yes', 'y', 'true', '1', 'positive') or v.startswith('yes'):
        return 'Yes'
    if v in ('no', 'n', 'false', '0', 'negative') or v.startswith('no'):
        return 'No'
    return 'Unknown'


In [ ]:
def load_progression_data():
    print("Loading radiology progression data...")
    df = pd.read_csv(PROGRESSION_PATH, low_memory=False)
    df['MRN'] = standardize_mrn(df['MRN'])
    df = df[df['MRN'].notna()].copy()
    df['START_DATE'] = pd.to_datetime(df['START_DATE'], errors='coerce')
    df = df[df['START_DATE'].notna()].copy()
    df['progression_label'] = df['PROGRESSION'].apply(normalize_progression_label)
    df['source_specific'] = df['SOURCE_SPECIFIC'].fillna('Unknown').str.strip()
    df['progression_prob'] = pd.to_numeric(df['PROGRESSION_PROBABILITY'], errors='coerce')
    n_total, n_pts = len(df), df['MRN'].nunique()
    print(f"  Loaded {n_total:,} radiology records across {n_pts:,} patients")
    for lbl, cnt in df['progression_label'].value_counts().items():
        print(f"  PROGRESSION={lbl}: {cnt:,} ({100*cnt/n_total:.1f}%)")
    return df


def load_llm_patient_calls():
    print("\nLoading pre-built LLM patient-level binary calls...")
    df = pd.read_csv(LLM_PATIENT_PATH, encoding='latin-1', low_memory=False)
    df.columns = df.columns.str.replace('\ufeff', '', regex=False).str.strip()
    mrn_col = next((c for c in df.columns if c.lower() in ('mrn', 'patient_id')), None)
    if mrn_col is None:
        raise ValueError(f"MRN column not found. Detected: {list(df.columns[:10])}")
    df['mrn'] = standardize_mrn(df[mrn_col])
    df = df[df['mrn'].notna()].copy()
    df.rename(columns={'adrenal insufficiency': 'adrenal_insufficiency', 'liver toxicity': 'liver_toxicity'}, inplace=True)
    avail_tox = [t for t in TOXICITY_COLUMNS if t in df.columns]
    if not avail_tox:
        raise ValueError(f"No toxicity columns found. Detected: {list(df.columns)}")
    for t in avail_tox:
        df[t] = pd.to_numeric(df[t], errors='coerce').fillna(0).astype(int)
    df['any_irae'] = (df[avail_tox].sum(axis=1) > 0).astype(int)
    print(f"  Loaded {len(df):,} patients | Any irAE: {df['any_irae'].sum():,} ({100*df['any_irae'].mean():.1f}%)")
    return df


def load_grade_data(valid_mrns):
    print("\nLoading grade data and assigning onset grades...")
    if not os.path.exists(GRADE_PATH):
        print(f"  WARNING: Grade file not found at {GRADE_PATH}. Returning zeros.")
        empty = pd.DataFrame({'mrn': list(valid_mrns)})
        for t in TOXICITY_COLUMNS:
            empty[f'{t}_onset_grade'] = 0
        empty['max_onset_grade'] = 0
        return empty
    df = pd.read_csv(GRADE_PATH, encoding='latin-1', low_memory=False)
    df.columns = df.columns.str.replace('\ufeff', '', regex=False).str.strip()
    df.rename(columns={'adrenal insufficiency': 'adrenal_insufficiency', 'liver toxicity': 'liver_toxicity'}, inplace=True)
    mrn_col = next((c for c in df.columns if c.lower() in ('mrn', 'patient_id')), None)
    if mrn_col is None:
        raise ValueError(f"MRN column not found in grade file. Columns: {list(df.columns[:10])}")
    df['mrn'] = standardize_mrn(df[mrn_col])
    df = df[df['mrn'].notna() & df['mrn'].isin(valid_mrns)].copy()
    df['window_start'] = pd.to_datetime(df['window_start'], errors='coerce')
    df = df[df['window_start'].notna()].sort_values(['mrn', 'window_start'])
    avail_tox = [t for t in TOXICITY_COLUMNS if t in df.columns]
    for t in avail_tox:
        df[t] = pd.to_numeric(df[t], errors='coerce').fillna(0).astype(int)
    onset_parts = []
    for t in avail_tox:
        masked = df[['mrn', t]].copy()
        masked.loc[masked[t] == 0, t] = np.nan
        first_pos = (masked.groupby('mrn')[t].first().fillna(0).astype(int).rename(f'{t}_onset_grade'))
        onset_parts.append(first_pos)
    grade_df = pd.concat(onset_parts, axis=1).reset_index()
    grade_df.columns.name = None
    onset_cols = [f'{t}_onset_grade' for t in avail_tox]
    grade_df['max_onset_grade'] = grade_df[onset_cols].max(axis=1).astype(int)
    all_mrns = pd.DataFrame({'mrn': list(valid_mrns)})
    grade_df = all_mrns.merge(grade_df, on='mrn', how='left').fillna(0)
    for col in onset_cols + ['max_onset_grade']:
        grade_df[col] = grade_df[col].astype(int)
    n_any = (grade_df['max_onset_grade'] > 0).sum()
    print(f"  Onset grades assigned for {len(grade_df):,} patients | {n_any:,} with any grade > 0")
    return grade_df


def load_lot_data(valid_mrns):
    print("\nLoading LOT data...")
    lot = pd.read_csv(LOT_DATA_PATH, encoding='latin-1', low_memory=False)
    lot.columns = lot.columns.str.strip()
    lot['MRN'] = standardize_mrn(lot['MRN'])
    lot = lot[lot['MRN'].notna() & lot['MRN'].isin(valid_mrns)].copy()
    lot['APR_START_DTE'] = pd.to_datetime(lot['APR_START_DTE'], errors='coerce')
    lot['APR_END_DTE'] = pd.to_datetime(lot['APR_END_DTE'], errors='coerce')
    lot = lot[lot['APR_START_DTE'].notna()].sort_values(['MRN', 'APR_START_DTE'])

    def _bool_col(df, candidates):
        col = next((c for c in candidates if c in df.columns), None)
        if col is None:
            return pd.Series(0, index=df.index)
        return df[col].fillna(0).apply(lambda x: 1 if str(x).strip().upper() in ('1', 'TRUE', 'YES', 'Y') else 0)

    lot['contains_ici'] = _bool_col(lot, ['CONTAINS_IMMUNO', 'contains_immuno', 'CONTAINS_ICI', 'contains_ici'])
    lot['has_ctla4'] = _bool_col(lot, ['CONTAINS_CTLA4', 'contains_ctla4', 'CONTAINS_CTLA4_IMMUNO', 'contains_ctla4_immuno'])
    lot['has_pdl1'] = _bool_col(lot, ['CONTAINS_NON_CTLA4_IMMUNO', 'contains_non_ctla4_immuno', 'CONTAINS_PDL1', 'contains_pdl1'])
    lot['has_chemo'] = _bool_col(lot, ['CONTAINS_CHEMO', 'contains_chemo'])
    lot['has_targeted'] = _bool_col(lot, ['CONTAINS_TARGETED', 'contains_targeted'])
    lot['has_hormone'] = _bool_col(lot, ['CONTAINS_HORMONE', 'contains_hormone'])
    lot['has_biologic'] = _bool_col(lot, ['CONTAINS_BIOLOGIC', 'contains_biologic'])

    first_lot = (lot.groupby('MRN').agg(
        first_lot_start=('APR_START_DTE', 'first'), lot1_end=('APR_END_DTE', 'first'),
        lot1_ici=('contains_ici', 'first'), lot1_pdl1=('has_pdl1', 'first'),
        lot1_ctla4=('has_ctla4', 'first'), lot1_chemo=('has_chemo', 'first'),
        lot1_targeted=('has_targeted', 'first'), lot1_hormone=('has_hormone', 'first'),
        lot1_biologic=('has_biologic', 'first')).reset_index())
    for col in ['lot1_ici', 'lot1_pdl1', 'lot1_ctla4', 'lot1_chemo', 'lot1_targeted', 'lot1_hormone', 'lot1_biologic']:
        first_lot[col] = first_lot[col].fillna(0).astype(int)
    n_ici = first_lot['lot1_ici'].sum()
    n_noici = (first_lot['lot1_ici'] == 0).sum()
    print(f"  {first_lot['MRN'].nunique():,} patients | LOT-1 ICI: {n_ici:,} | LOT-1 non-ICI: {n_noici:,}")
    return first_lot


def load_patient_data(valid_mrns):
    print("\nLoading patient data...")
    pat = pd.read_csv(PATIENTS_PATH, low_memory=False)
    pat.columns = pat.columns.str.replace('\ufeff', '', regex=False).str.strip()
    mrn_col = next((c for c in pat.columns if 'MRN' in c.upper()), None)
    death_col = next((c for c in pat.columns if 'DEATH' in c.upper() or 'DOD' in c.upper()), None)
    lfu_col = next((c for c in pat.columns if any(k in c.upper() for k in ('LAST_CONTACT', 'LAST_FU', 'LAST_FOLLOW', 'LFU'))), None)
    if mrn_col is None:
        raise ValueError(f"MRN column not found. Columns: {list(pat.columns[:10])}")
    pat['MRN'] = standardize_mrn(pat[mrn_col])
    pat = pat[pat['MRN'].notna() & pat['MRN'].isin(valid_mrns)].copy()
    pat['death_date'] = pd.to_datetime(pat[death_col], errors='coerce') if death_col else pd.NaT
    if lfu_col:
        pat['last_followup_date'] = pd.to_datetime(pat[lfu_col], errors='coerce')
    elif death_col:
        print("  WARNING: No last follow-up column found. Using death_date as proxy for deceased.")
        pat['last_followup_date'] = pat['death_date']
    else:
        pat['last_followup_date'] = pd.NaT
        print("  WARNING: No last follow-up date available. Censoring will be unreliable.")
    print(f"  {len(pat):,} patients | {pat['death_date'].notna().sum():,} deaths | {pat['last_followup_date'].notna().sum():,} with last FU date")
    return pat[['MRN', 'death_date', 'last_followup_date']].drop_duplicates('MRN')


def load_covariate_data(valid_mrns):
    print("\nLoading covariate data (age, sex, cancer type, BMI)...")
    if not os.path.exists(COVARS_PATH):
        print(f"  WARNING: Covariates file not found at {COVARS_PATH}.")
        return pd.DataFrame({'mrn': list(valid_mrns)})
    df = pd.read_csv(COVARS_PATH, encoding='latin-1', low_memory=False)
    df.columns = df.columns.str.replace('\ufeff', '', regex=False).str.strip()
    mrn_col = next((c for c in df.columns if c.lower() in ('mrn', 'patient_id')), None)
    if mrn_col is None:
        print(f"  WARNING: MRN column not found in covariates file.")
        return pd.DataFrame({'mrn': list(valid_mrns)})
    df['mrn'] = standardize_mrn(df[mrn_col])
    df = df[df['mrn'].notna() & df['mrn'].isin(valid_mrns)].copy()
    lot_col = next((c for c in df.columns if c.lower() in ('lot', 'lot_number', 'line')), None)
    if lot_col:
        df[lot_col] = pd.to_numeric(df[lot_col], errors='coerce')
        df = df[df[lot_col].notna()].copy()
        df = df.sort_values(['mrn', lot_col])
        # Line-1 ROW via idxmin -- not .groupby().first(), which takes the first non-null value
        # per column independently and can pull bmi/cancer_type from a later line when LOT-1
        # has a null. cancer_type is the Cox strata variable, so this is not cosmetic.
        idx = df.groupby('mrn')[lot_col].idxmin()
        df = df.loc[idx].reset_index(drop=True)
    else:
        print("  WARNING: no LOT column found in covariates file -- taking first row per patient.")
        df = df.drop_duplicates('mrn').reset_index(drop=True)
    assert df['mrn'].is_unique, 'more than one covariate row per patient'
    age_col = next((c for c in df.columns if 'age' in c.lower() and 'dx' in c.lower()), None)
    if age_col is None:
        age_col = next((c for c in df.columns if 'age' in c.lower()), None)
    df['age_at_dx'] = pd.to_numeric(df[age_col], errors='coerce') if age_col else np.nan
    sex_col = next((c for c in df.columns if c.lower() in ('gender', 'sex', 'gender_clean')), None)
    if sex_col:
        df['gender_male'] = (df[sex_col].fillna('').str.upper().str.startswith('M')).astype(float)
        df.loc[df[sex_col].isna(), 'gender_male'] = np.nan
    else:
        df['gender_male'] = np.nan
    ct_col = next((c for c in df.columns if 'cancer_type' in c.lower()), None)
    df['cancer_type'] = df[ct_col].fillna('Unknown') if ct_col else 'Unknown'
    bmi_col = next((c for c in df.columns if 'bmi' in c.lower()), None)
    if bmi_col:
        df['bmi'] = pd.to_numeric(df[bmi_col], errors='coerce')
        df.loc[(df['bmi'] < 10) | (df['bmi'] > 80), 'bmi'] = np.nan
    else:
        df['bmi'] = np.nan
    out = df[['mrn', 'age_at_dx', 'gender_male', 'cancer_type', 'bmi']].copy()
    print(f"  Covariates loaded for {len(out):,} patients | cancer types: {out['cancer_type'].nunique()}")
    return out


def load_batch_level_calls(valid_mrns):
    print("\nLoading batch-level LLM calls for OS analysis...")
    if not os.path.exists(LLM_BATCH_PATH):
        print(f"  WARNING: Batch-level file not found at {LLM_BATCH_PATH}.")
        return pd.DataFrame()
    df = pd.read_csv(LLM_BATCH_PATH, encoding='latin-1', low_memory=False)
    df.columns = df.columns.str.replace('\ufeff', '', regex=False).str.strip()
    df.rename(columns={'adrenal insufficiency': 'adrenal_insufficiency', 'liver toxicity': 'liver_toxicity'}, inplace=True)
    mrn_col = next((c for c in df.columns if c.lower() == 'mrn'), None)
    if mrn_col is None:
        print(f"  WARNING: MRN column not found in batch file.")
        return pd.DataFrame()
    df['mrn'] = standardize_mrn(df[mrn_col])
    df = df[df['mrn'].notna() & df['mrn'].isin(valid_mrns)].copy()
    df['window_start'] = pd.to_datetime(df['window_start'], errors='coerce')
    df = df[df['window_start'].notna()].sort_values(['mrn', 'window_start'])
    print(f"  Loaded {len(df):,} windows for {df['mrn'].nunique():,} patients")
    return df


def get_os_ae_timing(batch_df, grade_raw, valid_mrns):
    avail_tox = [t for t in TOXICITY_COLUMNS if t in batch_df.columns]
    g1_frames, g3_frames = [], []
    for t in avail_tox:
        pos = batch_df[batch_df[t] == 1][['mrn', 'window_start']].drop_duplicates('mrn')
        g1_frames.append(pos.rename(columns={'window_start': f'{t}_g1_onset'}))
        if grade_raw is not None and t in grade_raw.columns:
            g3 = grade_raw[pd.to_numeric(grade_raw[t], errors='coerce').fillna(0) >= 3]
            pos3 = g3[['mrn', 'window_start']].drop_duplicates('mrn')
            g3_frames.append(pos3.rename(columns={'window_start': f'{t}_g3_onset'}))
    onset = pd.DataFrame({'mrn': list(valid_mrns)})
    for frame in g1_frames + g3_frames:
        onset = onset.merge(frame.set_index('mrn'), left_on='mrn', right_index=True, how='left')
    g1_cols = [f'{t}_g1_onset' for t in avail_tox if f'{t}_g1_onset' in onset.columns]
    g3_cols = [f'{t}_g3_onset' for t in avail_tox if f'{t}_g3_onset' in onset.columns]
    if g1_cols:
        onset['any_ae_g1_onset'] = onset[g1_cols].min(axis=1)
    if g3_cols:
        onset['any_ae_g3_onset'] = onset[g3_cols].min(axis=1)
    return onset.set_index('mrn')


def build_first_progression(prog_df, strategy):
    if strategy == 'yes_only':
        pos = prog_df[prog_df['progression_label'] == 'Yes'].copy()
    elif strategy == 'yes_or_intermediate':
        pos = prog_df[prog_df['progression_label'].isin(['Yes', 'Intermediate'])].copy()
    elif strategy == 'probability_050':
        pos = prog_df[prog_df['progression_prob'] >= 0.50].copy()
    else:
        raise ValueError(f"Unrecognized strategy: {strategy}")
    return (pos.sort_values('START_DATE').groupby('MRN').agg(
        progression_date=('START_DATE', 'first'),
        progression_prob_at_first=('PROGRESSION_PROBABILITY', 'first'),
        n_progression_events=('START_DATE', 'count')).reset_index()
        .assign(progression_prob_at_first=lambda d: pd.to_numeric(d['progression_prob_at_first'], errors='coerce')))


## Build the analysis cohort (per cohort type: ICI / non-ICI / all)

This merges LLM predictions, LOT baseline covariates, patient death/follow-up dates, demographic
covariates, and progression data, then computes PFS fields (unused here) and OS fields (`os_days`,
`os_event`) used by the OS model below.


In [ ]:
def build_analysis_cohort(llm_calls, grade_df, baseline_df, pat_df, covar_df, first_prog, cohort_type, strategy_name):
    t0_col = 'first_lot_start'
    avail_tox = [t for t in TOXICITY_COLUMNS if t in llm_calls.columns]
    grade_cols = [f'{t}_onset_grade' for t in TOXICITY_COLUMNS if f'{t}_onset_grade' in grade_df.columns]
    print(f"\n  --- Merge diagnostics [{cohort_type} | {strategy_name}] ---")
    n0 = llm_calls['mrn'].nunique()
    cohort = llm_calls[['mrn', 'any_irae'] + avail_tox].merge(
        grade_df[['mrn'] + grade_cols + ['max_onset_grade']], on='mrn', how='left')
    for gc in grade_cols + ['max_onset_grade']:
        if gc in cohort.columns:
            cohort[gc] = cohort[gc].fillna(0).astype(int)
    cohort = cohort.merge(baseline_df.rename(columns={'MRN': 'mrn'}), on='mrn', how='inner')
    n1 = len(cohort)
    print(f"  Step 1 — After inner join with LOT data:   {n1:>8,}  (lost {n0-n1:,})")
    if cohort_type == 'ici':
        cohort = cohort[cohort['lot1_ici'] == 1].copy()
    elif cohort_type == 'non_ici':
        cohort = cohort[cohort['lot1_ici'] == 0].copy()
    n_filtered = len(cohort)
    print(f"  Step 1b — After cohort filter [{cohort_type}]: {n_filtered:>8,}")
    cohort = cohort[cohort[t0_col].notna()].copy()
    cohort.rename(columns={t0_col: 't0_date', 'lot1_end': 'lot1_end_date'}, inplace=True)
    cohort = cohort.merge(pat_df.rename(columns={'MRN': 'mrn'}), on='mrn', how='left')
    cohort = cohort.merge(covar_df, on='mrn', how='left')
    prog_cols = ['mrn', 'progression_date', 'progression_prob_at_first', 'n_progression_events']
    cohort = cohort.merge(first_prog.rename(columns={'MRN': 'mrn'})[prog_cols], on='mrn', how='left')
    n5 = len(cohort)
    cohort['days_to_prog_raw'] = (cohort['progression_date'] - cohort['t0_date']).dt.days
    n_pre_t0 = ((cohort['days_to_prog_raw'].notna()) & (cohort['days_to_prog_raw'] < 0)).sum()
    cohort.loc[cohort['days_to_prog_raw'] < 0, 'progression_date'] = pd.NaT
    cohort['days_to_prog'] = (cohort['progression_date'] - cohort['t0_date']).dt.days
    cohort['days_to_death'] = (cohort['death_date'] - cohort['t0_date']).dt.days
    cohort['days_to_lfu'] = (cohort['last_followup_date'] - cohort['t0_date']).dt.days
    print(f"  Pre-T0 progression nulled: {n_pre_t0:,} | rows: {n5:,}")
    cohort.loc[cohort['days_to_death'] < 0, 'death_date'] = pd.NaT
    cohort['days_to_death'] = (cohort['death_date'] - cohort['t0_date']).dt.days
    prog_days = cohort['days_to_prog'].fillna(np.inf).values
    death_days = cohort['days_to_death'].fillna(np.inf).values
    lfu_days = cohort['days_to_lfu'].fillna(0).values
    first_event = np.minimum(prog_days, death_days)
    has_event = (first_event <= lfu_days) & np.isfinite(first_event)
    cohort['pfs_days'] = np.where(has_event, first_event, np.maximum(lfu_days, 0))
    cohort['pfs_event'] = has_event.astype(int)
    cohort['pfs_months'] = cohort['pfs_days'] / 30.44
    cohort = cohort[cohort['pfs_days'] > 0].copy()
    n_final = len(cohort)
    # Cancer type: kept as a raw categorical column for Cox STRATIFICATION, not dummy-encoded
    # as a fixed effect. This gives each cancer type its own baseline hazard rather than
    # assuming one shared baseline shape shifted by a constant HR per type.
    if 'cancer_type' in cohort.columns:
        cohort['cancer_type'] = cohort['cancer_type'].fillna('Unknown').astype(str)
    cohort = cohort.drop(columns=['days_to_prog_raw'], errors='ignore')
    death_v = cohort['days_to_death'].fillna(np.inf).values
    lfu_v = cohort['days_to_lfu'].fillna(0).values
    has_os = np.isfinite(death_v) & (death_v <= lfu_v)
    cohort['os_days'] = np.where(has_os, death_v, np.maximum(lfu_v, 0))
    cohort['os_event'] = has_os.astype(int)
    cohort['os_months'] = cohort['os_days'] / 30.44
    if n_final > 0:
        n_irae = cohort['any_irae'].sum()
        print(f"  FINAL: {n_final:,} patients | AE+: {n_irae:,} ({100*n_irae/n_final:.1f}%) | "
              f"PFS events: {cohort['pfs_event'].sum():,} | OS events: {cohort['os_event'].sum():,}")
    return cohort


## OS time-varying Cox: covariates, time-varying rows, and the model fit


In [ ]:
def _get_cox_covariates(tv, cohort_type, extra_cols=None):
    # cancer_type deliberately excluded -- it's passed to CoxTimeVaryingFitter via strata=,
    # not included as a fixed-effect covariate here.
    if cohort_type == 'all':
        base = ['age_at_dx', 'gender_male', 'bmi', 'lot1_ici', 'lot1_chemo', 'lot1_targeted', 'lot1_hormone', 'lot1_biologic']
    else:
        base = ['age_at_dx', 'gender_male', 'bmi', 'lot1_chemo', 'lot1_targeted', 'lot1_hormone', 'lot1_biologic']
    candidates = base + (extra_cols or [])
    return [c for c in candidates if c in tv.columns and tv[c].nunique() >= 2 and pd.to_numeric(tv[c], errors='coerce').notna().sum() > 0]


def build_os_tv_rows(cohort, os_timing, tox, grade='g1'):
    onset_col = f'{tox}_{grade}_onset' if tox != 'any_ae' else f'any_ae_{grade}_onset'
    if onset_col not in os_timing.columns:
        return pd.DataFrame()
    static_candidates = ['age_at_dx', 'gender_male', 'bmi', 'lot1_ici', 'lot1_chemo', 'lot1_targeted', 'lot1_hormone', 'lot1_biologic']
    sel_cols = (['mrn', 't0_date', 'death_date', 'last_followup_date', 'cancer_type'] +
                [c for c in cohort.columns if c in static_candidates])
    c = cohort[sel_cols].copy()
    c['onset_date'] = c['mrn'].map(os_timing[onset_col])
    c['onset_months'] = (c['onset_date'] - c['t0_date']).dt.days / 30.44
    c['days_to_death'] = (c['death_date'] - c['t0_date']).dt.days.clip(lower=0)
    c['days_to_lfu'] = (c['last_followup_date'] - c['t0_date']).dt.days.clip(lower=0)
    death_v = c['days_to_death'].fillna(np.inf).values
    lfu_v = c['days_to_lfu'].fillna(0).values
    has_death = np.isfinite(death_v) & (death_v <= lfu_v)
    c['os_months'] = np.where(has_death, death_v, np.maximum(lfu_v, 0)) / 30.44
    c['os_event'] = has_death.astype(int)
    c = c[c['os_months'] > 0].copy()
    c.loc[c['onset_months'] <= 0, 'onset_months'] = np.nan
    c['has_onset'] = c['onset_months'].notna() & (c['onset_months'] < c['os_months'])
    static_cols = [col for col in c.columns if col in static_candidates]
    strata_cols = ['cancer_type']  # kept separate -- stratified, not adjusted for
    rows = []
    exposed = c[c['has_onset']].copy()
    unexposed = c[~c['has_onset']].copy()
    if len(exposed):
        pre = exposed.copy()
        pre['start'] = 0.0
        pre['stop'] = pre['onset_months']
        pre['os_event'] = 0
        pre['irae_exposed'] = 0
        rows.append(pre[['mrn', 'start', 'stop', 'os_event', 'irae_exposed'] + static_cols + strata_cols])
        post = exposed.copy()
        post['start'] = post['onset_months']
        post['stop'] = post['os_months']
        post['irae_exposed'] = 1
        rows.append(post[['mrn', 'start', 'stop', 'os_event', 'irae_exposed'] + static_cols + strata_cols])
    if len(unexposed):
        unexposed['start'] = 0.0
        unexposed['stop'] = unexposed['os_months']
        unexposed['irae_exposed'] = 0
        rows.append(unexposed[['mrn', 'start', 'stop', 'os_event', 'irae_exposed'] + static_cols + strata_cols])
    if not rows:
        return pd.DataFrame()
    tv = pd.concat(rows, ignore_index=True)
    return tv[tv['stop'] > tv['start']].copy()


def _fit_tv_cox_worker(tv, formula, event_col, tox_display, label):
    if tv is None or len(tv) == 0:
        return None
    try:
        ctv = CoxTimeVaryingFitter(penalizer=0.1)
        ctv.fit(tv, id_col='mrn', start_col='start', stop_col='stop', event_col=event_col,
                formula=formula, strata=['cancer_type'])
        if 'irae_exposed' not in ctv.params_.index:
            return None
        hr = float(np.exp(ctv.params_['irae_exposed']))
        ci_cols = ctv.confidence_intervals_.columns
        ci_lower = float(np.exp(ctv.confidence_intervals_.loc['irae_exposed', ci_cols[0]]))
        ci_upper = float(np.exp(ctv.confidence_intervals_.loc['irae_exposed', ci_cols[1]]))
        p = float(ctv.summary.loc['irae_exposed', 'p'])
        return {'hr': hr, 'ci_lower': ci_lower, 'ci_upper': ci_upper, 'p_value': p,
                'tox_display': tox_display, 'label': label, 'n': tv['mrn'].nunique(),
                'n_exposed': tv[tv['irae_exposed'] == 1]['mrn'].nunique(), 'n_events': int(tv[event_col].sum())}
    except Exception as exc:
        return {'error': str(exc), 'tox_display': tox_display, 'label': label}


def compute_os_results(cohort, os_timing, label, cohort_type):
    n_events_cohort = int(cohort['death_date'].notna().sum())
    out = []
    for tox in TOXICITY_COLUMNS:
        for grade in ('g1', 'g3'):
            tv = build_os_tv_rows(cohort, os_timing, tox, grade)
            if len(tv) < 50:
                continue
            cov_cols = _get_cox_covariates(tv, cohort_type)
            keep = ['mrn', 'start', 'stop', 'os_event', 'irae_exposed'] + cov_cols + ['cancer_type']
            tv_clean = tv[[c for c in keep if c in tv.columns]].dropna()
            tv_clean = tv_clean[tv_clean['stop'] > tv_clean['start']].copy()
            # Strata with fewer than 2 patients contribute ~nothing to the irae_exposed estimate
            # but can still slow/destabilize the fit -- drop them rather than including dead weight.
            strat_sizes = tv_clean.groupby('cancer_type')['mrn'].nunique()
            valid_strata = strat_sizes[strat_sizes >= 2].index
            tv_clean = tv_clean[tv_clean['cancer_type'].isin(valid_strata)].copy()
            if tv_clean['os_event'].sum() < MIN_EVENTS_KM or tv_clean['irae_exposed'].nunique() < 2:
                continue
            formula = 'irae_exposed + ' + ' + '.join(cov_cols) if cov_cols else 'irae_exposed'
            row_label = f"{TOXICITY_DISPLAY.get(tox, tox)} ({GRADE_TIER_LABEL[grade]})"
            res = _fit_tv_cox_worker(tv_clean, formula, 'os_event', row_label, label)
            if res is None:
                continue
            if 'error' in res:
                print(f"  OS TV-Cox [{label}] {res['tox_display']}: FAILED - {res['error']}")
                continue
            out.append({'ae': res['tox_display'], 'hr': res['hr'], 'ci_lower': res['ci_lower'],
                        'ci_upper': res['ci_upper'], 'p_value': res['p_value'], 'n': res['n'],
                        'n_exposed': res['n_exposed'], 'n_events': n_events_cohort})
            print(f"  OS TV-Cox [{label}] {res['tox_display']}: HR={res['hr']:.2f} "
                  f"(95% CI {res['ci_lower']:.2f}-{res['ci_upper']:.2f}), p={res['p_value']:.4f}, "
                  f"n_exposed={res['n_exposed']:,}")
    return out


## Forest plot renderer

Nature text tiers (7pt axis label / 6pt tick labels / 5pt annotation text), no `bbox_inches='tight'`.
`save_path=None` renders inline without writing a file (used for the non-ICI / all-patients QC
plots); passing a path saves the PDF (used only for the ICI panel).


In [ ]:
def render_forest_fig2(results, endpoint, label, save_path=None):
    if not results:
        print(f"  Forest skipped ({endpoint}, {label}): no results")
        return
    df = pd.DataFrame(results)
    df = df[df['n_exposed'] >= 10].copy()
    if len(df) == 0:
        print(f"  Forest skipped ({endpoint}, {label}): no rows with n_exposed >= 10")
        return
    df['_base'] = df['ae'].str.replace(r'\s*\(Grade.*', '', regex=True).str.strip()
    df['_tier'] = np.where(df['ae'].str.contains('Grade 3', na=False), 'G3+', 'G1+')
    disp_to_key = {TOXICITY_DISPLAY.get(t, t): t for t in TOXICITY_COLUMNS}
    disp_to_key['Any irAE'] = 'any_ae'
    order_index = {t: i for i, t in enumerate(_TOX_ORDER)}
    df['_ord'] = df['_base'].map(lambda b: order_index.get(disp_to_key.get(b, b), 99))
    df['_tord'] = (df['_tier'] == '3+').astype(int)
    df = df.sort_values(['_ord', '_tord']).reset_index(drop=True)
    n = len(df)

    fig, ax = plt.subplots(figsize=(3.889, 2.25))
    y = np.arange(n - 1, -1, -1)
    for i, (_, row) in enumerate(df.iterrows()):
        hr = row['hr']
        lo = max(row['ci_lower'], 0.1)
        hi = min(row['ci_upper'], 10.0)
        col = '#E63946' if row['p_value'] < 0.05 else '#888888'
        ax.plot([lo, hi], [y[i], y[i]], color=col, lw=2.2, solid_capstyle='round')
        ax.plot(max(min(hr, 10), 0.1), y[i], 'o', color=col, ms=6, markeredgecolor='white', markeredgewidth=0.5, zorder=5)
    ax.axvline(1, color='#404040', ls='--', lw=1.0, zorder=0)
    labels = [f"{r['_base']} {r['_tier']}" for _, r in df.iterrows()]
    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=6)
    ax.set_xscale('log')
    ax.set_xlim(0.1, 10)
    ax.set_xticks([0.1, 0.25, 0.5, 1, 2, 4, 10])
    ax.set_xticklabels(['0.1', '0.25', '0.5', '1', '2', '4', '10'], fontsize=6)
    ax.set_xlabel('Hazard Ratio (log scale)', fontsize=7)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.tick_params(axis='y', length=0)
    ax.set_ylim(-0.7, n - 0.3)
    plt.tight_layout()

    plt.show()

    if save_path is not None:
        fig.savefig(save_path, dpi=450)
        print(f"  Saved {endpoint} forest [{label}] ({n} rows): {os.path.basename(save_path)}")
    plt.close(fig)


## Run: build all three cohorts, compute OS results for each, save only the ICI forest

Non-ICI and All-patients forests render inline (for QC) but are not written to disk — only the ICI cohort is saved as the panel deliverable. A combined CSV of
all three cohorts' OS results is saved for reference regardless.


In [ ]:
prog_df = load_progression_data()
llm_calls = load_llm_patient_calls()
valid_mrns = set(llm_calls['mrn'].dropna())
grade_df = load_grade_data(valid_mrns)
baseline_df = load_lot_data(valid_mrns)
pat_df = load_patient_data(valid_mrns)
covar_df = load_covariate_data(valid_mrns)
batch_df = load_batch_level_calls(valid_mrns)

grade_raw = None
if os.path.exists(GRADE_PATH):
    grade_raw = pd.read_csv(GRADE_PATH, encoding='latin-1', low_memory=False)
    grade_raw.columns = grade_raw.columns.str.replace('\ufeff', '', regex=False).str.strip()
    grade_raw.rename(columns={'adrenal insufficiency': 'adrenal_insufficiency', 'liver toxicity': 'liver_toxicity'}, inplace=True)
    mrn_col_g = next((c for c in grade_raw.columns if c.lower() == 'mrn'), None)
    if mrn_col_g:
        grade_raw['mrn'] = standardize_mrn(grade_raw[mrn_col_g])
        grade_raw['window_start'] = pd.to_datetime(grade_raw['window_start'], errors='coerce')
        grade_raw = grade_raw[grade_raw['mrn'].notna() & grade_raw['mrn'].isin(valid_mrns)].sort_values(['mrn', 'window_start'])

os_timing = get_os_ae_timing(batch_df, grade_raw, valid_mrns) if len(batch_df) else None
first_prog = build_first_progression(prog_df, strategy=PROGRESSION_STRATEGY)

labels = {'ici': 'LOT-1 ICI', 'non_ici': 'LOT-1 Non-ICI', 'all': 'All Patients'}
all_os_results = []

for ck in ['ici', 'non_ici', 'all']:
    label = labels[ck]
    print("\n" + "=" * 60)
    print(f"COHORT: {label}")
    print("=" * 60)
    cohort = build_analysis_cohort(llm_calls, grade_df, baseline_df, pat_df, covar_df, first_prog, ck, label)
    if len(cohort) < 30:
        print(f"  Cohort too small (n={len(cohort)}); skipping.")
        continue
    if os_timing is None:
        print("  OS skipped: batch-level calls unavailable, no OS AE timing.")
        continue

    os_results = compute_os_results(cohort, os_timing, label, ck)
    if os_results:
        for r in os_results:
            r['cohort'] = ck
            r['cohort_label'] = label
        all_os_results.extend(os_results)
        save_path = PDF_OUT if ck == 'ici' else None
        render_forest_fig2(os_results, 'OS', label, save_path=save_path)
    else:
        print(f"  No OS results for cohort {label}.")

if all_os_results:
    results_df = pd.DataFrame(all_os_results)
    results_df.to_csv(CSV_OUT, index=False)
    print(f"\nSaved: {os.path.basename(CSV_OUT)} ({len(results_df)} rows, all cohorts)")
else:
    print("\nNo OS results across any cohort -- nothing saved.")
